# Session 2: Signals to Decisions

## Few-Shot Learning for Nowcasting Institutional Crises

This lab uses a synthetic but intentionally imperfect monitoring problem.

- We observe a daily corpus of **18,000 headlines** across **five countries**.
- We want to detect three institutional risks: **coup**, **term-limit evasion**, and **judiciary weakening**.
- We begin with a transparent dictionary benchmark, then move to SetFit, tune the alert threshold, and aggregate article-level scores to the country-month level.


## Storyline

We keep **2025** out for testing and use the earlier period for development.

1. Split the article corpus into `train`, `calibration`, and `test`
2. Start with a **dictionary benchmark**
3. Train **SetFit** on a small few-shot label set
4. Tune an article-level alert threshold on the calibration sample
5. Freeze that threshold and evaluate the 2025 holdout year
6. Aggregate the tuned article scores into monthly nowcasting indicators


In [ ]:
import io
import contextlib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from datasets import Dataset
from setfit import SetFitModel, Trainer, TrainingArguments, logging as setfit_logging
setfit_logging.set_verbosity_error()

from FewShotX import DictionaryScorer, configure_notebook
tqdm, _ = configure_notebook(theme="whitegrid", progress_bar="rich")

from sklearn.decomposition import TruncatedSVD
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import (
    average_precision_score,
    classification_report,
    confusion_matrix,
    precision_recall_curve,
    roc_curve,
    auc,
)
from sklearn.model_selection import train_test_split

np.random.seed(42)


df_corpus = pd.read_csv("datasets/df_corpus.csv", parse_dates=["date"])
df_labels = pd.read_csv("datasets/df_labels.csv")
df_target = pd.read_csv("datasets/df_target.csv")

df_corpus["month"] = df_corpus["date"].dt.to_period("M").astype(str)
df_corpus["year"] = df_corpus["date"].dt.year

label_order = ["other", "coup", "term_limit_evasion", "judiciary_weakening"]
event_order = ["coup", "term_limit_evasion", "judiciary_weakening"]
severity_map = {"other": 0, "judiciary_weakening": 1, "term_limit_evasion": 2, "coup": 3}

last_year = int(df_corpus["year"].max())
dev_pool = df_corpus.query("year < @last_year").copy()
test_df = df_corpus.query("year == @last_year").copy()

dev_train, calib_df = train_test_split(
    dev_pool,
    test_size=0.35,
    random_state=42,
    stratify=dev_pool["true_label"],
)

split_map = pd.concat(
    [
        dev_train[["article_id"]].assign(split="train"),
        calib_df[["article_id"]].assign(split="calibration"),
        test_df[["article_id"]].assign(split="test"),
    ],
    ignore_index=True,
)

df_corpus = df_corpus.merge(split_map, on="article_id", how="left")

print("Corpus shape:", df_corpus.shape)
print("Few-shot labels shape:", df_labels.shape)
print("Target shape:", df_target.shape)
print("Test year:", last_year)


In [ ]:
print("Split sizes")
display(df_corpus["split"].value_counts().to_frame("n"))

print("\nCountries in the corpus")
display(df_corpus["country"].value_counts().to_frame("n"))

print("\nFew-shot labels per class")
display(df_labels["label"].value_counts().reindex(label_order))

print("\nArticle-level class balance in the full corpus")
display(
    (
        df_corpus["true_label"]
        .value_counts(normalize=True)
        .rename("share")
        .reindex(label_order)
        .mul(100)
        .round(2)
        .to_frame()
    )
)

print("\nHard-case mix in the corpus")
display(df_corpus["hard_case_type"].value_counts().to_frame("n"))

In [ ]:
def fb_score(precision, recall, beta=0.5):
    beta_sq = beta ** 2
    denom = beta_sq * precision + recall
    if denom == 0:
        return 0.0
    return (1 + beta_sq) * precision * recall / denom


def binary_alert_metrics(y_true_labels, y_pred_labels):
    y_true = (pd.Series(y_true_labels) != "other").astype(int).to_numpy()
    y_pred = (pd.Series(y_pred_labels) != "other").astype(int).to_numpy()

    tp = int(((y_true == 1) & (y_pred == 1)).sum())
    fp = int(((y_true == 0) & (y_pred == 1)).sum())
    fn = int(((y_true == 1) & (y_pred == 0)).sum())

    precision = tp / (tp + fp) if (tp + fp) else 0.0
    recall = tp / (tp + fn) if (tp + fn) else 0.0
    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) else 0.0
    f0_5 = fb_score(precision, recall, beta=0.5)
    return precision, recall, f1, f0_5


def article_summary(df, pred_col, method, stage, threshold_mode):
    report = classification_report(
        df["true_label"],
        df[pred_col],
        labels=label_order,
        output_dict=True,
        zero_division=0,
    )
    alert_precision, alert_recall, alert_f1, alert_f0_5 = binary_alert_metrics(
        df["true_label"],
        df[pred_col],
    )
    return {
        "method": method,
        "stage": stage,
        "threshold_mode": threshold_mode,
        "macro_f1": report["macro avg"]["f1-score"],
        "weighted_f1": report["weighted avg"]["f1-score"],
        "alert_precision": alert_precision,
        "alert_recall": alert_recall,
        "alert_f1": alert_f1,
        "alert_f0_5": alert_f0_5,
    }


def apply_threshold(df, candidate_col, score_col, threshold, pred_col):
    out = df.copy()
    out[pred_col] = np.where(out[score_col] >= threshold, out[candidate_col], "other")
    return out


def tune_alert_threshold(df, candidate_col, score_col, thresholds):
    rows = []
    for threshold in thresholds:
        temp_pred = np.where(df[score_col] >= threshold, df[candidate_col], "other")
        precision, recall, f1, f0_5 = binary_alert_metrics(df["true_label"], temp_pred)
        rows.append(
            {
                "threshold": threshold,
                "alert_precision": precision,
                "alert_recall": recall,
                "alert_f1": f1,
                "alert_f0_5": f0_5,
            }
        )
    tuning = pd.DataFrame(rows)
    best_threshold = tuning.sort_values(
        ["alert_f0_5", "alert_precision", "alert_recall"],
        ascending=False,
    ).iloc[0]["threshold"]
    return float(best_threshold), tuning


def sample_up_to(df, group_col, n, random_state=42):
    parts = []
    for _, grp in df.groupby(group_col):
        parts.append(grp.sample(n=min(n, len(grp)), random_state=random_state))
    return pd.concat(parts, ignore_index=True)

## 1. Inspect The Hard Cases

Before fitting any model, it helps to look at the headlines that are designed to confuse a simple classifier.


In [ ]:
(
    df_corpus.query("hard_case_type != 'none'")
    .groupby("hard_case_type", group_keys=False)
    .head(3)[["country", "date", "headline", "true_label", "hard_case_type"]]
    .reset_index(drop=True)
)

In [ ]:
plt.figure(figsize=(10, 4))
hard_counts = (
    df_corpus["hard_case_type"]
    .value_counts()
    .rename_axis("hard_case_type")
    .reset_index(name="count")
)
sns.barplot(data=hard_counts, x="hard_case_type", y="count", color="#2f6db3")
plt.xticks(rotation=35, ha="right")
plt.title("Hard-case composition of the article corpus")
plt.tight_layout()
plt.show()

## 2. Dictionary Benchmark

We start simple. The dictionary method is transparent and fast, but it is vulnerable to false positives from historical references, foreign-country mentions, and broad legal vocabulary.


In [ ]:
crisis_dictionary = {
    "coup": ["coup", "junta", "mutiny", "takeover", "putsch", "soldiers", "guards", "detain", "barracks"],
    "term_limit_evasion": ["term", "third", "mandate", "eligibility", "referendum", "amendment", "constitution", "charter"],
    "judiciary_weakening": ["judge", "judges", "court", "judicial", "magistrates", "prosecutors", "tribunal", "justice"],
}

scorer = DictionaryScorer(dictionaries=crisis_dictionary, model_name="en_core_web_sm")
df_dict = scorer.score_df(
    df_corpus[
        ["article_id", "date", "country", "month", "year", "split", "headline", "true_label", "hard_case_type", "difficulty"]
    ].copy(),
    text_col="headline",
)

dict_score_cols = list(crisis_dictionary)
df_dict["dict_candidate_label"] = df_dict[dict_score_cols].idxmax(axis=1)
df_dict["dict_crisis_score"] = df_dict[dict_score_cols].max(axis=1).astype(float)
dict_baseline_threshold = 1.0
df_dict["dict_pred_baseline"] = np.where(
    df_dict["dict_crisis_score"] >= dict_baseline_threshold,
    df_dict["dict_candidate_label"],
    "other",
)

df_dict.head()

In [ ]:
dict_train = df_dict.query("split == 'train'").copy()

dict_train_summary = article_summary(
    dict_train,
    pred_col="dict_pred_baseline",
    method="dictionary",
    stage="train",
    threshold_mode="baseline",
)
display(pd.DataFrame([dict_train_summary]))

cm_dict_train = confusion_matrix(
    dict_train["true_label"],
    dict_train["dict_pred_baseline"],
    labels=label_order,
    normalize="true",
)

plt.figure(figsize=(7, 5))
sns.heatmap(
    cm_dict_train,
    annot=True,
    fmt=".2f",
    cmap="Oranges",
    xticklabels=label_order,
    yticklabels=label_order,
)
plt.title("Dictionary baseline: normalized confusion matrix on train")
plt.xlabel("Predicted label")
plt.ylabel("True label")
plt.show()

## 3. Train SetFit On A Few-Shot Label Set

SetFit should be stronger because it can learn context rather than relying on exact keywords.


In [ ]:
label_to_id = {label: idx for idx, label in enumerate(label_order)}
id_to_label = {idx: label for label, idx in label_to_id.items()}

train_labels = df_labels.assign(label_id=lambda d: d["label"].map(label_to_id))
train_ds = Dataset.from_pandas(
    train_labels[["headline", "label_id"]].rename(columns={"headline": "text", "label_id": "label"}),
    preserve_index=False,
)

def load_setfit_model_quietly(model_name):
    hidden_logs = io.StringIO()
    with contextlib.redirect_stdout(hidden_logs), contextlib.redirect_stderr(hidden_logs):
        model = SetFitModel.from_pretrained(model_name)
    return model


model = load_setfit_model_quietly("sentence-transformers/all-MiniLM-L6-v2")

args = TrainingArguments(
    batch_size=8,
    num_epochs=2,
    num_iterations=3,
    body_learning_rate=2e-5,
    head_learning_rate=1e-2,
    logging_steps=5,
    report_to="none",
)

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=train_ds,
)

print("Training examples:", len(train_labels))
display(train_labels["label"].value_counts().reindex(label_order))
trainer.train()

## 4. Score The Article Corpus With SetFit

We score the full article stream in batches.


In [ ]:
texts = df_corpus["headline"].tolist()
batch_size = 256
pred_prob_batches = []

for start in tqdm(
    range(0, len(texts), batch_size),
    desc="Scoring articles with SetFit",
    dynamic_ncols=True,
    leave=False,
    colour="#2f6db3",
):
    batch_texts = texts[start : start + batch_size]
    pred_prob_batches.append(np.asarray(model.predict_proba(batch_texts)))

pred_proba = np.vstack(pred_prob_batches)
prob_cols = [f"p_{label}" for label in label_order]

df_scores = pd.concat(
    [
        df_corpus.reset_index(drop=True),
        pd.DataFrame(pred_proba, columns=prob_cols),
    ],
    axis=1,
)

event_prob_cols = [f"p_{label}" for label in event_order]
df_scores["setfit_candidate_label"] = (
    df_scores[event_prob_cols]
    .idxmax(axis=1)
    .str.replace("p_", "", regex=False)
)
df_scores["setfit_crisis_score"] = df_scores[event_prob_cols].max(axis=1)
setfit_baseline_threshold = 0.50
df_scores["setfit_pred_baseline"] = np.where(
    df_scores["setfit_crisis_score"] >= setfit_baseline_threshold,
    df_scores["setfit_candidate_label"],
    "other",
)

df_scores.head()


In [ ]:
setfit_train = df_scores.query("split == 'train'").copy()

setfit_train_summary = article_summary(
    setfit_train,
    pred_col="setfit_pred_baseline",
    method="setfit",
    stage="train",
    threshold_mode="baseline",
)

display(pd.DataFrame([dict_train_summary, setfit_train_summary]).sort_values("method"))

cm_setfit_train = confusion_matrix(
    setfit_train["true_label"],
    setfit_train["setfit_pred_baseline"],
    labels=label_order,
    normalize="true",
)

plt.figure(figsize=(7, 5))
sns.heatmap(
    cm_setfit_train,
    annot=True,
    fmt=".2f",
    cmap="Blues",
    xticklabels=label_order,
    yticklabels=label_order,
)
plt.title("SetFit baseline: normalized confusion matrix on train")
plt.xlabel("Predicted label")
plt.ylabel("True label")
plt.show()

The train-sample comparison is only a first look. The real question is what happens once we use a separate **calibration** sample to decide how aggressive our alert threshold should be.

A useful way to explain the evaluation metrics is with a simple alerting example:

- Suppose the system issues **10 alerts**.
- If **6** of them are correct, precision is **0.60**.
- If there were **12** real crisis articles in total, recall is **0.50**.

From there:

- **F1** balances precision and recall equally.
- **F0.5** puts more weight on precision.

That policy choice matters here. In a humanitarian early-warning setting, false positives can trigger costly diplomatic attention or operational responses, so we may prefer a threshold that is more conservative than the one that only maximizes recall.


## 5. Tune The Article-Level Alert Threshold

We now calibrate a simple **alert threshold**:

- the classifier first picks the most likely crisis type
- the threshold decides whether that score is large enough to issue any alert at all

For the dictionary, we deliberately restrict the calibration rule to a very interpretable choice:

- baseline: **at least one** crisis keyword match
- tuned: **at least two** crisis keyword matches

This keeps the tuned dictionary rule easy to explain in class and avoids awkward rules such as requiring three or more matched keywords in a single headline.


In [ ]:
dict_tuned_threshold, dict_tuning = tune_alert_threshold(
    df_dict.query("split == 'calibration'"),
    candidate_col="dict_candidate_label",
    score_col="dict_crisis_score",
    thresholds=[1.0, 2.0],
)

setfit_tuned_threshold, setfit_tuning = tune_alert_threshold(
    df_scores.query("split == 'calibration'"),
    candidate_col="setfit_candidate_label",
    score_col="setfit_crisis_score",
    thresholds=np.linspace(0.50, 0.99, 500),
)

print("Dictionary baseline threshold:", dict_baseline_threshold)
print("Dictionary tuned threshold:", round(dict_tuned_threshold, 4))
print("SetFit baseline threshold:", setfit_baseline_threshold)
print("SetFit tuned threshold:", round(setfit_tuned_threshold, 4))

In [ ]:
df_dict = apply_threshold(
    df_dict,
    candidate_col="dict_candidate_label",
    score_col="dict_crisis_score",
    threshold=dict_tuned_threshold,
    pred_col="dict_pred_tuned",
)

df_scores = apply_threshold(
    df_scores,
    candidate_col="setfit_candidate_label",
    score_col="setfit_crisis_score",
    threshold=setfit_tuned_threshold,
    pred_col="setfit_pred_tuned",
)

calibration_comparison = pd.DataFrame(
    [
        article_summary(df_dict.query("split == 'calibration'"), "dict_pred_baseline", "dictionary", "calibration", "baseline"),
        article_summary(df_dict.query("split == 'calibration'"), "dict_pred_tuned", "dictionary", "calibration", "tuned"),
        article_summary(df_scores.query("split == 'calibration'"), "setfit_pred_baseline", "setfit", "calibration", "baseline"),
        article_summary(df_scores.query("split == 'calibration'"), "setfit_pred_tuned", "setfit", "calibration", "tuned"),
    ]
)

calibration_comparison.sort_values(["method", "threshold_mode"])

In [ ]:
tuning_plot = pd.concat(
    [
        dict_tuning.assign(method="dictionary"),
        setfit_tuning.assign(method="setfit"),
    ],
    ignore_index=True,
)

plt.figure(figsize=(9, 4))
sns.lineplot(
    data=tuning_plot,
    x="threshold",
    y="alert_f0_5",
    hue="method",
    linewidth=2,
)
plt.title("Calibration objective: article-level alert F0.5")
plt.tight_layout()
plt.show()

By construction, the tuned threshold is chosen to improve **article-level alert quality** on the calibration sample, not necessarily every possible metric. This is intentional: in early warning, the threshold is often about policy priorities rather than pure classification accuracy.


## 6. Evaluate The 2025 Holdout Year

We now freeze the tuned threshold and move to the **2025 test year**.


In [ ]:
test_comparison = pd.DataFrame(
    [
        article_summary(df_dict.query("split == 'test'"), "dict_pred_baseline", "dictionary", "test", "baseline"),
        article_summary(df_dict.query("split == 'test'"), "dict_pred_tuned", "dictionary", "test", "tuned"),
        article_summary(df_scores.query("split == 'test'"), "setfit_pred_baseline", "setfit", "test", "baseline"),
        article_summary(df_scores.query("split == 'test'"), "setfit_pred_tuned", "setfit", "test", "tuned"),
    ]
)

test_comparison.sort_values(["method", "threshold_mode"])

In [ ]:
test_dict = df_dict.query("split == 'test'").copy()
test_setfit = df_scores.query("split == 'test'").copy()

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

cm_dict_base = confusion_matrix(
    test_dict["true_label"],
    test_dict["dict_pred_baseline"],
    labels=label_order,
    normalize="true",
)
sns.heatmap(
    cm_dict_base,
    annot=True,
    fmt=".2f",
    cmap="Oranges",
    xticklabels=label_order,
    yticklabels=label_order,
    ax=axes[0],
)
axes[0].set_title("Dictionary baseline")
axes[0].set_xlabel("Predicted label")
axes[0].set_ylabel("True label")

cm_setfit_base = confusion_matrix(
    test_setfit["true_label"],
    test_setfit["setfit_pred_baseline"],
    labels=label_order,
    normalize="true",
)
sns.heatmap(
    cm_setfit_base,
    annot=True,
    fmt=".2f",
    cmap="Blues",
    xticklabels=label_order,
    yticklabels=label_order,
    ax=axes[1],
)
axes[1].set_title("SetFit baseline")
axes[1].set_xlabel("Predicted label")
axes[1].set_ylabel("True label")

cm_setfit_tuned = confusion_matrix(
    test_setfit["true_label"],
    test_setfit["setfit_pred_tuned"],
    labels=label_order,
    normalize="true",
)
sns.heatmap(
    cm_setfit_tuned,
    annot=True,
    fmt=".2f",
    cmap="Greens",
    xticklabels=label_order,
    yticklabels=label_order,
    ax=axes[2],
)
axes[2].set_title("SetFit tuned")
axes[2].set_xlabel("Predicted label")
axes[2].set_ylabel("True label")

plt.tight_layout()
plt.show()

print(f"SetFit tuned: article-level classification report on the {last_year} test year")
print(
    classification_report(
        test_setfit["true_label"],
        test_setfit["setfit_pred_tuned"],
        labels=label_order,
        digits=3,
        zero_division=0,
    )
)

## 7. ROC And Precision-Recall Curves

Confusion matrices show one operating point. ROC and PR curves show how that operating point moves as we vary the alert threshold.

Here the true score-based methods are the dictionary crisis score and the SetFit crisis score. The baseline and tuned SetFit versions share the same curve; what changes is the chosen operating point.


In [ ]:
from sklearn.metrics import roc_auc_score

y_test_bin = (test_setfit["true_label"] != "other").astype(int).to_numpy()
dict_score_test = test_dict["dict_crisis_score"].astype(float).to_numpy()
setfit_score_test = test_setfit["setfit_crisis_score"].astype(float).to_numpy()

def operating_point(y_true_labels, y_pred_labels):
    y_true = (pd.Series(y_true_labels) != "other").astype(int).to_numpy()
    y_pred = (pd.Series(y_pred_labels) != "other").astype(int).to_numpy()

    tp = ((y_true == 1) & (y_pred == 1)).sum()
    fp = ((y_true == 0) & (y_pred == 1)).sum()
    tn = ((y_true == 0) & (y_pred == 0)).sum()
    fn = ((y_true == 1) & (y_pred == 0)).sum()

    tpr = tp / (tp + fn) if (tp + fn) else 0.0
    fpr = fp / (fp + tn) if (fp + tn) else 0.0
    precision = tp / (tp + fp) if (tp + fp) else 0.0
    recall = tpr
    return fpr, tpr, recall, precision

fpr_dict, tpr_dict, _ = roc_curve(y_test_bin, dict_score_test)
fpr_setfit, tpr_setfit, _ = roc_curve(y_test_bin, setfit_score_test)
roc_auc_dict = auc(fpr_dict, tpr_dict)
roc_auc_setfit = auc(fpr_setfit, tpr_setfit)

prec_dict, rec_dict, _ = precision_recall_curve(y_test_bin, dict_score_test)
prec_setfit, rec_setfit, _ = precision_recall_curve(y_test_bin, setfit_score_test)
ap_dict = average_precision_score(y_test_bin, dict_score_test)
ap_setfit = average_precision_score(y_test_bin, setfit_score_test)

points = {
    "Dictionary baseline": operating_point(test_dict["true_label"], test_dict["dict_pred_baseline"]),
    "SetFit baseline": operating_point(test_setfit["true_label"], test_setfit["setfit_pred_baseline"]),
    "SetFit tuned": operating_point(test_setfit["true_label"], test_setfit["setfit_pred_tuned"]),
}

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

axes[0].plot(fpr_dict, tpr_dict, color="#d95f02", linewidth=2, label=f"Dictionary score (AUC={roc_auc_dict:.3f})")
axes[0].plot(fpr_setfit, tpr_setfit, color="#1b9e77", linewidth=2, label=f"SetFit score (AUC={roc_auc_setfit:.3f})")
for label, (fpr, tpr, recall, precision) in points.items():
    axes[0].scatter(fpr, tpr, s=70, label=label)
axes[0].plot([0, 1], [0, 1], linestyle="--", color="gray", linewidth=1)
axes[0].set_title("ROC curve on the 2025 test set")
axes[0].set_xlabel("False positive rate")
axes[0].set_ylabel("True positive rate")
axes[0].legend(loc="lower right")

axes[1].plot(rec_dict, prec_dict, color="#d95f02", linewidth=2, label=f"Dictionary score (AP={ap_dict:.3f})")
axes[1].plot(rec_setfit, prec_setfit, color="#1b9e77", linewidth=2, label=f"SetFit score (AP={ap_setfit:.3f})")
for label, (fpr, tpr, recall, precision) in points.items():
    axes[1].scatter(recall, precision, s=70, label=label)
axes[1].set_title("Precision-recall curve on the 2025 test set")
axes[1].set_xlabel("Recall")
axes[1].set_ylabel("Precision")
axes[1].legend(loc="lower left")

plt.tight_layout()
plt.show()

## 8. Error Patterns And Article Clusters

We can now inspect where the tuned SetFit model still fails, even after threshold calibration.


In [ ]:
error_by_case = (
    df_scores.query("split == 'test'")
    .assign(setfit_error=lambda d: d["setfit_pred_tuned"] != d["true_label"])
    .groupby("hard_case_type", as_index=False)
    .agg(
        n_articles=("headline", "size"),
        setfit_error_rate=("setfit_error", "mean"),
    )
    .merge(
        df_dict.query("split == 'test'")
        .assign(dict_error=lambda d: d["dict_pred_tuned"] != d["true_label"])
        .groupby("hard_case_type", as_index=False)
        .agg(dict_error_rate=("dict_error", "mean")),
        on="hard_case_type",
        how="left",
    )
    .sort_values("setfit_error_rate", ascending=False)
)

error_by_case

In [ ]:
error_long = error_by_case.melt(
    id_vars=["hard_case_type", "n_articles"],
    value_vars=["setfit_error_rate", "dict_error_rate"],
    var_name="method",
    value_name="error_rate",
)

plt.figure(figsize=(11, 4))
sns.barplot(data=error_long, x="hard_case_type", y="error_rate", hue="method")
plt.xticks(rotation=35, ha="right")
plt.title("Test-set error rate by hard-case type")
plt.tight_layout()
plt.show()

In [ ]:
viz_source = df_scores.query("split == 'test'").copy()
viz_sample = (
    pd.concat(
        [
            sample_up_to(viz_source, "true_label", 100, random_state=42),
            viz_source.query("hard_case_type != 'none'").sample(
                n=min(220, len(viz_source.query("hard_case_type != 'none'"))),
                random_state=42,
            ),
        ],
        ignore_index=True,
    )
    .drop_duplicates(subset=["article_id"])
    .reset_index(drop=True)
)

tfidf = TfidfVectorizer(max_features=2500, ngram_range=(1, 2), min_df=3)
X_viz = tfidf.fit_transform(viz_sample["headline"])
coords = TruncatedSVD(n_components=2, random_state=42).fit_transform(X_viz)
viz_sample["svd_1"] = coords[:, 0]
viz_sample["svd_2"] = coords[:, 1]

plt.figure(figsize=(10, 7))
sns.scatterplot(
    data=viz_sample,
    x="svd_1",
    y="svd_2",
    hue="true_label",
    style="is_hard_case",
    alpha=0.68,
    s=55,
)
plt.title("Test-year article map from TF-IDF + SVD")
plt.xlabel("Component 1")
plt.ylabel("Component 2")
plt.legend(bbox_to_anchor=(1.02, 1), loc="upper left")
plt.tight_layout()
plt.show()

In [ ]:
false_positives = df_scores.query("split == 'test' and true_label == 'other' and setfit_pred_tuned != 'other'")
false_negatives = df_scores.query("split == 'test' and true_label != 'other' and setfit_pred_tuned == 'other'")

print("SetFit tuned false positives")
display(
    false_positives[
        ["country", "headline", "setfit_pred_tuned", "hard_case_type", "difficulty", "setfit_crisis_score"]
    ].head(12)
)

print("\nSetFit tuned false negatives")
display(
    false_negatives[
        ["country", "headline", "true_label", "hard_case_type", "difficulty", "setfit_crisis_score"]
    ].head(12)
)

## 9. Aggregate Tuned Article Scores To The Month Level

The final step is to aggregate the **tuned SetFit model** to the country-month level and see whether it tracks the planted event calendar.


In [ ]:
df_test_setfit = df_scores.query("split == 'test'").copy()
df_test_setfit["setfit_row_severity_prob"] = (
    3 * df_test_setfit["p_coup"]
    + 2 * df_test_setfit["p_term_limit_evasion"]
    + 1 * df_test_setfit["p_judiciary_weakening"]
)
df_test_setfit["setfit_row_severity_alert"] = df_test_setfit["setfit_pred_tuned"].map(severity_map)

monthly_setfit = (
    df_test_setfit.groupby(["country", "month"], as_index=False)
    .agg(
        coup_prob=("p_coup", "mean"),
        term_limit_evasion_prob=("p_term_limit_evasion", "mean"),
        judiciary_weakening_prob=("p_judiciary_weakening", "mean"),
        severity_prob_index=("setfit_row_severity_prob", "mean"),
        severity_alert_index=("setfit_row_severity_alert", "mean"),
        alert_share=("setfit_pred_tuned", lambda s: (s != "other").mean()),
    )
    .merge(df_target, on=["country", "month"], how="left")
    .sort_values(["country", "month"])
)

monthly_setfit.head()

In [ ]:
severity_panel = monthly_setfit.pivot(index="country", columns="month", values="severity_prob_index")
plt.figure(figsize=(14, 4))
sns.heatmap(severity_panel, cmap="YlOrRd", linewidths=0.4)
plt.title("SetFit severity-weighted monthly risk index in the 2024 test year")
plt.xlabel("Month")
plt.ylabel("Country")
plt.tight_layout()
plt.show()

In [ ]:
country_to_plot = "Bolivia"
panel = monthly_setfit.query("country == @country_to_plot").copy()
panel["month_ts"] = pd.to_datetime(panel["month"] + "-01")

fig, axes = plt.subplots(4, 1, figsize=(12, 11), sharex=True)

for ax, event in zip(axes[:3], event_order):
    ax.plot(
        panel["month_ts"],
        panel[f"{event}_prob"],
        marker="o",
        linewidth=2,
        color="#2f6db3",
        label="Predicted probability",
    )
    ax.axhline(
        setfit_baseline_threshold,
        color="gray",
        linestyle="--",
        linewidth=1,
        label="Article baseline threshold",
    )
    ax.axhline(
        setfit_tuned_threshold,
        color="black",
        linestyle=":",
        linewidth=1,
        label="Article tuned threshold",
    )
    ax.set_ylabel("Pred prob")
    ax.set_title(f"{country_to_plot}: {event}")

    ax_right = ax.twinx()
    ax_right.step(
        panel["month_ts"],
        panel[event],
        where="mid",
        color="firebrick",
        linewidth=2,
        label="Planted event",
    )
    ax_right.set_ylim(-0.05, 1.05)
    ax_right.set_ylabel("Actual")

    lines_left, labels_left = ax.get_legend_handles_labels()
    lines_right, labels_right = ax_right.get_legend_handles_labels()
    ax.legend(lines_left + lines_right, labels_left + labels_right, loc="upper right")

ax = axes[3]
ax.plot(
    panel["month_ts"],
    panel["severity_prob_index"],
    marker="o",
    linewidth=2,
    color="#1b9e77",
    label="Predicted severity index",
)
ax.set_ylabel("Pred severity")
ax.set_title(f"{country_to_plot}: severity-weighted index")

ax_right = ax.twinx()
ax_right.step(
    panel["month_ts"],
    panel["severity_score"],
    where="mid",
    color="#d95f02",
    linewidth=2,
    label="Planted severity score",
)
ax_right.set_ylabel("Actual severity")

lines_left, labels_left = ax.get_legend_handles_labels()
lines_right, labels_right = ax_right.get_legend_handles_labels()
ax.legend(lines_left + lines_right, labels_left + labels_right, loc="upper right")

plt.tight_layout()
plt.show()

## Suggested Discussion Questions

1. Why is threshold tuning better framed here as an **alerting** problem rather than a full multiclass optimization problem?
2. Which hard-case type hurts the dictionary benchmark the most?
3. Why might a policymaker prefer the tuned SetFit threshold over the baseline threshold, even if recall falls?
4. What is gained by aggregating to the month level, and what is lost?
5. How could we use the severity index as an input to a downstream nowcasting or early-warning model?
